In [ ]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    RAIZ = os.path.abspath("..")
else:
    RAIZ = os.path.abspath(".")

sys.path.append(RAIZ)

import pandas as pd
import matplotlib.pyplot as plt

from src.detectores import (
    contar_flags_estatisticas,
    rodar_isolation_forest,
    rodar_lof,
)

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

df = pd.read_csv(os.path.join(RAIZ, "data", "processed", "obras_features.csv"))

# Camada 1: flags estatísticas em todos os indicadores.
# Corte em >=2 flags: uma flag isolada pode ser acaso estatístico,
# duas ou mais em indicadores diferentes já é padrão.
df["qtd_flags"] = contar_flags_estatisticas(df)
df["flag_estatistica"] = df["qtd_flags"] >= 2

# Camada 2: Isolation Forest
df["score_iforest"], df["flag_iforest"] = rodar_isolation_forest(df)

# Camada 3: LOF
df["score_lof"], df["flag_lof"] = rodar_lof(df)

print("Obras flagradas por cada camada:")
print(f"  flags estatísticas (>=2): {df['flag_estatistica'].sum()}")
print(f"  isolation forest:         {df['flag_iforest'].sum()}")
print(f"  lof:                      {df['flag_lof'].sum()}")

Obras flagradas por cada camada:
  flags estatísticas (>=2): 169
  isolation forest:         120
  lof:                      120


In [ ]:
gabarito = pd.read_csv(os.path.join(RAIZ, "data", "raw", "obras_gabarito.csv"))[["id_obra", "anomalia"]]
av = df.merge(gabarito, on="id_obra")
av["eh_anomalia"] = av["anomalia"] != "nenhuma"


def avaliar(flag, nome):
    """Compara uma flag binária com o gabarito e devolve as métricas."""
    vp = (flag & av["eh_anomalia"]).sum()       # verdadeiros positivos
    fp = (flag & ~av["eh_anomalia"]).sum()      # falsos positivos
    fn = (~flag & av["eh_anomalia"]).sum()      # falsos negativos
    recall = vp / (vp + fn)
    precisao = vp / (vp + fp)
    return {
        "detector": nome,
        "flagradas": int(flag.sum()),
        "verdadeiros_positivos": int(vp),
        "falsos_positivos": int(fp),
        "recall": f"{recall:.0%}",
        "precisao": f"{precisao:.0%}",
    }


tabela = pd.DataFrame([
    avaliar(av["flag_estatistica"], "flags estatisticas (>=2)"),
    avaliar(av["flag_iforest"], "isolation forest"),
    avaliar(av["flag_lof"], "lof"),
])
tabela

,detector,flagradas,verdadeiros_positivos,falsos_positivos,recall,precisao
0,flags estatisticas (>=2),169,120,49,100%,71%
1,isolation forest,120,106,14,88%,88%
2,lof,120,115,5,96%,96%
